# BPIC-17 Decision Mining

## 1. Objective

This notebook prepares the decision mining material for the assignment. It identifies branching activities in the complete-filtered BPIC-17 log and trains interpretable classifiers to explain which case and prefix attributes are associated with selected next-step decisions.

The goal is not to optimize predictive performance at all costs. The goal is to create academically defensible, reproducible material for the report: decision point definition, feature construction without future leakage, model quality, and interpretation.


## 2. Imports and Configuration

The notebook uses the same complete-event abstraction as process discovery. All paths are relative to the repository root.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pm4py

warnings.filterwarnings("ignore", message="Install the optional requirement.*")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() or (candidate / "requirements.txt").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "BPI Challenge 2017.xes"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

CASE_ID_COL = "case:concept:name"
ACTIVITY_COL = "concept:name"
TIMESTAMP_COL = "time:timestamp"
LIFECYCLE_COL = "lifecycle:transition"
EVENT_ORIGIN_COL = "EventOrigin"

RANDOM_SEED = 42

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Project root resolved.")
print(f"Data file exists: {DATA_PATH.exists()}")

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

DECISION_FIGURES_DIR = FIGURES_DIR / "decision_mining"
DECISION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_DECISION_POINTS = ["A_Validating", "O_Returned"]
TOP_TARGET_CLASSES = 5
TOP_PREFIX_ACTIVITIES = 12
TEST_SIZE = 0.30

def save_figure(stem: str, directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(directory / f"{stem}.png", dpi=300)
    plt.savefig(directory / f"{stem}.pdf")
    plt.show()


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()[:60]



## 3. Load Complete-Filtered Event Log

Only `lifecycle:transition == "complete"` events are used. This follows the process discovery notebook and treats completed activities as the business-level event abstraction.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "BPIC-17 event log not found. Place it at data/BPI Challenge 2017.xes."
    )

raw_log = pm4py.read_xes(str(DATA_PATH), show_progress_bar=False)
df = raw_log if isinstance(raw_log, pd.DataFrame) else pm4py.convert_to_dataframe(raw_log)
df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce", utc=True)

complete_log = (
    df[df[LIFECYCLE_COL].eq("complete")]
    .sort_values([CASE_ID_COL, TIMESTAMP_COL])
    .reset_index(drop=True)
    .copy()
)

print(f"Raw events: {len(df):,}")
print(f"Complete events: {len(complete_log):,}")
print(f"Complete cases: {complete_log[CASE_ID_COL].nunique():,}")
print(f"Complete activities: {complete_log[ACTIVITY_COL].nunique():,}")


## 4. Identify Candidate Decision Points

A decision point is approximated as an activity with multiple observed next activities. The candidate table helps justify why the selected decision points are suitable for decision mining.


In [ ]:
complete_log["next_activity"] = complete_log.groupby(CASE_ID_COL)[ACTIVITY_COL].shift(-1)
complete_log["previous_activity"] = complete_log.groupby(CASE_ID_COL)[ACTIVITY_COL].shift(1).fillna("START")

branch_counts = (
    complete_log.dropna(subset=["next_activity"])
    .groupby([ACTIVITY_COL, "next_activity"])
    .size()
    .reset_index(name="transition_count")
)

branch_summary = (
    branch_counts.groupby(ACTIVITY_COL)
    .agg(
        total_outgoing_events=("transition_count", "sum"),
        number_of_successors=("next_activity", "nunique"),
        dominant_successor_count=("transition_count", "max"),
    )
    .reset_index()
)
branch_summary["dominant_successor_share"] = (
    branch_summary["dominant_successor_count"] / branch_summary["total_outgoing_events"]
)
branch_summary = branch_summary.sort_values(
    ["number_of_successors", "total_outgoing_events"], ascending=False
)

branch_summary.to_csv(RESULTS_DIR / "decision_points.csv", index=False)
display(branch_summary.head(20))

for decision_point in SELECTED_DECISION_POINTS:
    print(f"\nSuccessors after {decision_point}:")
    display(
        branch_counts[branch_counts[ACTIVITY_COL].eq(decision_point)]
        .sort_values("transition_count", ascending=False)
        .head(10)
    )

def save_figure(stem: str, directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(directory / f"{stem}.png", dpi=300)
    plt.savefig(directory / f"{stem}.pdf")
    plt.show()


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()[:60]



## 5. Prefix Feature Construction

For each occurrence of a selected decision activity, features are computed from information available up to that event: case attributes, elapsed time, prefix length, previous activity, event-origin counts, and cumulative counts of frequent activities. The target is the next activity after the decision point.


In [ ]:
case_start = complete_log.groupby(CASE_ID_COL)[TIMESTAMP_COL].transform("min")
complete_log["elapsed_days"] = (complete_log[TIMESTAMP_COL] - case_start).dt.total_seconds() / 86400
complete_log["prefix_length"] = complete_log.groupby(CASE_ID_COL).cumcount() + 1

activity_count_columns = []
top_prefix_activities = complete_log[ACTIVITY_COL].value_counts().head(TOP_PREFIX_ACTIVITIES).index.tolist()
for activity in top_prefix_activities:
    column = f"count_activity_{safe_name(activity)}"
    complete_log[column] = complete_log[ACTIVITY_COL].eq(activity).astype(int).groupby(complete_log[CASE_ID_COL]).cumsum()
    activity_count_columns.append(column)

origin_count_columns = []
if EVENT_ORIGIN_COL in complete_log.columns:
    for origin in sorted(complete_log[EVENT_ORIGIN_COL].dropna().unique()):
        column = f"count_origin_{safe_name(origin)}"
        complete_log[column] = complete_log[EVENT_ORIGIN_COL].eq(origin).astype(int).groupby(complete_log[CASE_ID_COL]).cumsum()
        origin_count_columns.append(column)

case_attribute_columns = [CASE_ID_COL, "case:LoanGoal", "case:ApplicationType", "case:RequestedAmount"]
case_attributes = complete_log[case_attribute_columns].drop_duplicates(subset=[CASE_ID_COL])

base_numeric_features = ["case:RequestedAmount", "elapsed_days", "prefix_length"] + activity_count_columns + origin_count_columns
base_categorical_features = ["case:LoanGoal", "case:ApplicationType", "previous_activity"]

print("Numeric features:", base_numeric_features)
print("Categorical features:", base_categorical_features)


## 6. Train Interpretable Decision Models

For each selected decision point, the target keeps the five most frequent next activities and maps all remaining successors to `Other`. A shallow decision tree is used as the primary interpretable model, with a random forest as a comparison model.


In [ ]:
def prepare_decision_dataset(decision_point: str) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    rows = complete_log[
        complete_log[ACTIVITY_COL].eq(decision_point) & complete_log["next_activity"].notna()
    ].copy()
    rows = rows.merge(case_attributes, on=CASE_ID_COL, how="left", suffixes=("", "_case"))

    target_counts = rows["next_activity"].value_counts()
    top_targets = set(target_counts.head(TOP_TARGET_CLASSES).index)
    rows["target_next_activity"] = rows["next_activity"].where(rows["next_activity"].isin(top_targets), "Other")

    feature_columns = base_numeric_features + base_categorical_features
    X = rows[feature_columns].copy()
    y = rows["target_next_activity"].astype(str)
    return X, y, target_counts


def make_classifier_pipeline(model):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), base_numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), base_categorical_features),
        ]
    )
    return Pipeline([("preprocess", preprocessor), ("model", model)])


def evaluate_classifier(name: str, model_name: str, y_true: pd.Series, y_pred: np.ndarray) -> dict:
    return {
        "decision_point": name,
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "test_size": len(y_true),
        "classes": json.dumps(sorted(pd.Series(y_true).unique().tolist())),
    }

metrics_rows = []
importance_rows = []
rules_text_parts = []

dataset_overview_rows = []

for decision_point in SELECTED_DECISION_POINTS:
    X, y, original_target_counts = prepare_decision_dataset(decision_point)
    dataset_overview_rows.append(
        {
            "decision_point": decision_point,
            "instances": len(y),
            "classes_after_grouping": y.nunique(),
            "original_successors": original_target_counts.shape[0],
            "target_distribution": json.dumps(y.value_counts().to_dict()),
        }
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )

    tree_pipeline = make_classifier_pipeline(
        DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=100,
            class_weight="balanced",
            random_state=RANDOM_SEED,
        )
    )
    tree_pipeline.fit(X_train, y_train)
    tree_pred = tree_pipeline.predict(X_test)
    metrics_rows.append(evaluate_classifier(decision_point, "DecisionTreeClassifier", y_test, tree_pred))

    feature_names = tree_pipeline.named_steps["preprocess"].get_feature_names_out()
    tree_model = tree_pipeline.named_steps["model"]
    rules_text_parts.append(f"\n## {decision_point}\n")
    rules_text_parts.append(
        export_text(tree_model, feature_names=list(feature_names), max_depth=4)
    )

    plt.figure(figsize=(22, 10))
    plot_tree(
        tree_model,
        feature_names=feature_names,
        class_names=tree_model.classes_,
        filled=True,
        rounded=True,
        fontsize=8,
        max_depth=4,
    )
    plt.title(f"Decision Tree for Next Activity after {decision_point}")
    save_figure(f"decision_tree_{safe_name(decision_point)}", DECISION_FIGURES_DIR)

    labels = sorted(y.unique())
    cm = confusion_matrix(y_test, tree_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(10, 8))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(f"Decision Tree Confusion Matrix: {decision_point}")
    save_figure(f"confusion_matrix_tree_{safe_name(decision_point)}", DECISION_FIGURES_DIR)

    rf_pipeline = make_classifier_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=50,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_SEED,
        )
    )
    rf_pipeline.fit(X_train, y_train)
    rf_pred = rf_pipeline.predict(X_test)
    metrics_rows.append(evaluate_classifier(decision_point, "RandomForestClassifier", y_test, rf_pred))

    rf_features = rf_pipeline.named_steps["preprocess"].get_feature_names_out()
    rf_importances = rf_pipeline.named_steps["model"].feature_importances_
    feature_importance = (
        pd.DataFrame({"feature": rf_features, "importance": rf_importances})
        .sort_values("importance", ascending=False)
        .head(20)
    )
    for _, row in feature_importance.iterrows():
        importance_rows.append(
            {
                "decision_point": decision_point,
                "feature": row["feature"],
                "importance": row["importance"],
            }
        )

    plt.figure(figsize=(10, 7))
    plt.barh(feature_importance["feature"][::-1], feature_importance["importance"][::-1])
    plt.title(f"Random Forest Feature Importance: {decision_point}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    save_figure(f"feature_importance_{safe_name(decision_point)}", DECISION_FIGURES_DIR)

decision_dataset_overview = pd.DataFrame(dataset_overview_rows)
decision_metrics = pd.DataFrame(metrics_rows)
decision_feature_importance = pd.DataFrame(importance_rows)

RESULTS_DIR.joinpath("decision_tree_rules.txt").write_text("\n".join(rules_text_parts), encoding="utf-8")
decision_dataset_overview.to_csv(RESULTS_DIR / "decision_mining_dataset_overview.csv", index=False)
decision_metrics.to_csv(RESULTS_DIR / "decision_mining_metrics.csv", index=False)
decision_feature_importance.to_csv(RESULTS_DIR / "decision_mining_feature_importance.csv", index=False)

display(decision_dataset_overview)
display(decision_metrics)

def save_figure(stem: str, directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(directory / f"{stem}.png", dpi=300)
    plt.savefig(directory / f"{stem}.pdf")
    plt.show()


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()[:60]



## 7. Save Decision Mining Summary

The summary file is intermediate material for the later report. It documents the chosen decision points, target definitions, model results, and limitations.


In [ ]:
summary_lines = [
    "# Decision Mining Summary",
    "",
    "## Scope",
    "",
    "This intermediate summary documents decision mining on the complete-filtered BPIC-17 event log. The analysis uses branching activities as decision points and predicts the next activity from case attributes and prefix features available before the next step occurs.",
    "",
    "## Selected Decision Points",
    "",
]

for _, row in decision_dataset_overview.iterrows():
    summary_lines.append(
        f"- `{row['decision_point']}`: {row['instances']:,} instances, "
        f"{row['classes_after_grouping']} target classes after grouping rare successors as `Other`."
    )

summary_lines.extend(["", "## Model Metrics", ""])
summary_lines.append(decision_metrics.to_markdown(index=False) if False else "")

# Build a small Markdown table without optional tabulate dependency.
columns = ["decision_point", "model", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
summary_lines.append("| " + " | ".join(columns) + " |")
summary_lines.append("| " + " | ".join(["---"] * len(columns)) + " |")
for _, row in decision_metrics[columns].iterrows():
    formatted = []
    for col in columns:
        value = row[col]
        formatted.append(f"{value:.4f}" if isinstance(value, float) else str(value))
    summary_lines.append("| " + " | ".join(formatted) + " |")

summary_lines.extend([
    "",
    "## Interpretation Notes",
    "",
    "- The decision trees provide interpretable rules that can be referenced in the report.",
    "- Random forests are used only as a comparison model and for feature importance, not as the main explanatory artifact.",
    "- Class imbalance is present because some successors dominate each decision point.",
    "- Rare successor activities are grouped as `Other`; this improves stability but reduces target granularity.",
    "- Features are restricted to case attributes and prefix information to avoid future leakage.",
    "",
    "## Generated Files",
    "",
    "- `results/decision_points.csv`",
    "- `results/decision_mining_dataset_overview.csv`",
    "- `results/decision_mining_metrics.csv`",
    "- `results/decision_mining_feature_importance.csv`",
    "- `results/decision_tree_rules.txt`",
    "- `figures/decision_mining/decision_tree_*.pdf/png`",
    "- `figures/decision_mining/confusion_matrix_tree_*.pdf/png`",
    "- `figures/decision_mining/feature_importance_*.pdf/png`",
])

(RESULTS_DIR / "decision_mining_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")
print("Saved decision mining summary.")


## 8. Report Notes

Decision mining material prepared here can support the report section on decisions in the discovered process. The strongest report artifacts are the branching candidate table, selected decision point definitions, shallow decision trees, confusion matrices, and discussion of limitations.
